In [ ]:
!pip install  pdfplumber faiss-cpu sentence-transformers

In [ ]:
import os
folder_path = "/content"
for file in os.listdir(folder_path):
  if file.endswith(".pdf"):
    os.remove(os.path.join(folder_path, file))
print("Old files removed")

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pdfplumber

def extract_text_from_pdf(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    return text

In [ ]:
import re

def extract_sections(text):
    sections = {
        "skills": "",
        "experience": "",
        "projects": "",
        "certifications": ""
    }

    text_lower = text.lower()


    patterns = {
        "skills": ["skills", "technical skills", "key skills"],
        "experience": ["experience", "work experience", "professional experience"],
        "projects": ["projects", "personal projects"],
        "certifications": ["certifications", "certificates"]
    }

    for key, keywords in patterns.items():
        for kw in keywords:
            pattern = rf"{kw}(.+?)(\n[A-Z ]{{3,}}|\Z)"
            match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            if match:
                sections[key] = match.group(1).strip()
                break

    return sections

In [ ]:
resume_texts = {}

for file_name in uploaded.keys():
    text = extract_text_from_pdf(file_name)
    resume_texts[file_name] = text

In [ ]:
def clean_text(text):
    text = text.lower()
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text[:2000]

In [ ]:
structured_data = {}

for file_name, text in resume_texts.items():
    structured_data[file_name] = extract_sections(text)

In [ ]:
cleaned_docs = []

for file_name, sections in structured_data.items():
    combined_text = clean_text(" ".join(sections.values()))

    cleaned_docs.append({
        "file": file_name,
        "text": combined_text
    })

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device ='cpu')

In [ ]:
import numpy as np

documents = []
embeddings = []

texts = [doc["text"] for doc in cleaned_docs]
documents = [doc["file"] for doc in cleaned_docs]

embeddings = embed_model.encode(
    texts,
    batch_size=4,
    show_progress_bar=True
)

In [ ]:
import numpy as np
import faiss

embeddings = np.array(embeddings).astype('float32')

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [ ]:
query = input("Enter Job Description: ")

query_embedding = embed_model.encode([query]).astype('float32')

D, I = index.search(query_embedding, k=min(5, len(documents)))

print("\n Top Matching resumes:\n")

for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):

    score = 1 / (1 + dist)
    percentage = round(score * 100, 2)

    print(f"Rank {rank}: {documents[idx]}")
    pc=round(percentage,2)
    print(f"Match Score: {pc}%")
    print("-" * 40)